# UM-Bridge with QMC.jl

Using QMC.jl to evaluate the [UM-Bridge Cantilever Beam Function](https://um-bridge-benchmarks.readthedocs.io/en/docs/forward-benchmarks/muq-beam-propagation.html) and approximate the expectation with respect to a uniform random variable.

Based on the Python QMCPy demo `umbridge.ipynb`.

## Prerequisites

1. **UMBridge.jl** — Julia client for UM-Bridge models:
   ```julia
   import Pkg; Pkg.add("UMBridge")
   ```

2. **Docker** — to run the benchmark model server:
   ```bash
   docker run -d --name muqbp -p 4243:4243 linusseelinger/benchmark-muq-beam-propagation:latest
   ```

## Imports

In [ ]:
using QMC
import QMC: Uniform
using Statistics
# UMBridge.jl is loaded automatically by UMBridgeWrapper if installed
# import Pkg; Pkg.add("UMBridge")  # uncomment if not yet installed
const HAVE_UMBRIDGE = Base.find_package("UMBridge") !== nothing
global umbridge_ready = false
global integrand = nothing
global result = nothing

## Start Docker Container

Run the UM-Bridge cantilever beam benchmark model.  See the
[UM-Bridge Documentation](https://um-bridge-benchmarks.readthedocs.io/)
for other available models.

In [ ]:
# Run this in a terminal (or uncomment the line below):
# run(`docker run --name muqbp -d -it -p 4243:4243 linusseelinger/benchmark-muq-beam-propagation:latest`)
println("Make sure the UM-Bridge container is running on port 4243.")
println("Start with:")
println("  docker run --name muqbp -d -p 4243:4243 linusseelinger/benchmark-muq-beam-propagation:latest")

## Problem Setup

Initialize a QMC sampler and distribution.  The cantilever beam model takes
3-dimensional input representing material/load parameters, uniformly
distributed in [1, 1.05].

In [ ]:
# Discrete distribution and true measure
sampler = DigitalNetB2(3; seed=7, randomize="LMS_DS", graycode=false)
distribution = Uniform(sampler; lower_bound=1.0, upper_bound=1.05)

println("Sampler: ", sampler)
println("Distribution: ", distribution)

## Create the UM-Bridge Integrand

The `UMBridgeWrapper` connects to the running Docker model via HTTP.
The `config` dict is passed to the model on each evaluation call.

In [ ]:
# Connect to UM-Bridge model
if HAVE_UMBRIDGE
    umbridge_config = Dict{String,Any}("d" => 3)
    global integrand = UMBridgeWrapper(distribution;
        url = "http://localhost:4243",
        model_name = "forward",
        config = umbridge_config)

    global umbridge_ready = integrand._model !== nothing
    println(integrand)
    if !umbridge_ready
        println("Skipping UM-Bridge evaluation cells because the UM-Bridge server is unavailable.")
    end
else
    println("UMBridge.jl is not installed; skipping UM-Bridge evaluation cells.")
end

## Model Evaluation

Generate QMC samples and evaluate the model directly.

In [ ]:
if umbridge_ready
    try
        let x_eval = gen_samples(sampler, 16)
            println("Sample points shape: ", size(x_eval))

            y_eval = evaluate(integrand, x_eval)

            println("Output shape: ", size(y_eval))
            println("Output type:  ", typeof(y_eval))
            println()
            println("First 4 outputs:")
            for i in 1:min(4, size(y_eval, 1))
                if ndims(y_eval) == 1
                    println("  y[$i] = $(round(y_eval[i], digits=4))")
                else
                    println("  y[$i,:] = $(round.(y_eval[i,:], digits=4))")
                end
            end
        end
    catch err
        global umbridge_ready = false
        println("Skipping remaining UM-Bridge cells: $(sprint(showerror, err))")
    end
else
    println("Skipping model evaluation because UM-Bridge is unavailable.")
end

## Automatically Approximate the Expectation

Use `CubQMCNetG` to adaptively integrate the UM-Bridge model output,
doubling the sample size until the error tolerance is met.

In [ ]:
if umbridge_ready
    try
        let sc_local = CubQMCNetG(integrand; abs_tol=2.5e-2)
            global result = integrate(sc_local)
        end

        println("Solution: ", result.solution)
        println("Error bound: ", result.data[:error_bound])
        println("Samples used: ", result.data[:n])
        println("Converged: ", result.data[:converged])
        println("Time: ", round(result.data[:time_integrate], digits=2), "s")
    catch err
        global umbridge_ready = false
        println("Skipping integration because the UM-Bridge server is unavailable: $(sprint(showerror, err))")
    end
else
    println("Skipping integration because UM-Bridge is unavailable.")
end

## Visualize the Solution

The cantilever beam model returns the displacement u(x) at multiple points
along the beam.  Plot the solution.

In [ ]:
if umbridge_ready
    sol = result.solution
    if isa(sol, AbstractVector)
        println("Beam displacement profile ($(length(sol)) points):")
        println()
        max_val = maximum(abs.(sol))
        for (i, v) in enumerate(sol)
            bar_len = round(Int, abs(v) / max(max_val, 1e-10) * 40)
            bar = repeat("█", bar_len)
            println("  x=$(lpad(i, 3)):  $(lpad(round(v, digits=2), 8))  $bar")
        end
    else
        println("Scalar solution: $(round(sol, digits=6))")
    end
else
    println("Skipping solution visualization because UM-Bridge is unavailable.")
end

## Parallel Evaluation

For expensive models, QMC.jl can be combined with Julia's native threading.
Use `Threads.@threads` in a custom evaluate loop for parallel HTTP requests.

In [ ]:
if umbridge_ready
    println("Available Julia threads: $(Threads.nthreads())")
    println()

    n_eval = 32
    x_par = gen_samples(sampler, n_eval)

    t_seq = @elapsed begin
        y_seq = evaluate(integrand, x_par)
    end
    println("Sequential ($n_eval evals): $(round(t_seq, digits=2))s")

    # Note: For true parallel UM-Bridge calls, you can use:
    # Threads.@threads for i in 1:n_eval
    #     y_par[i,:] = single_evaluate(integrand, x_par[i:i,:])
    # end
    # This sends HTTP requests in parallel to the UM-Bridge server.
else
    println("Skipping parallel evaluation example because UM-Bridge is unavailable.")
end

## Shut Down Docker Container

Clean up when done.

In [ ]:
# Run in terminal:
# docker rm -f muqbp
println("To stop the container:")
println("  docker rm -f muqbp")